# 3. Anonymization

Three passes on every title and text (see `anonymization_ner.py`): links and e-mail addresses → `[URL]`, `[EMAIL]`;
the outlet's own name written from its domain → `[OUTLET]`; then a standard named-entity recognizer, GLiNER (zero-shot
NER, `urchade/gliner_large-v2.1`), with the labels "news outlet" → `[OUTLET]` and "journalist" → `[AUTHOR]` (every
outlet and every journalist, not only the publisher). Nothing else changes, no rewriting.
Check: articles that still contain the name of their own outlet.

In [1]:
THRESHOLD = 0.5

In [2]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())
import anonymization_ner as AX

AX.THRESHOLD = THRESHOLD
art = pd.read_csv(C.OUT / "articles.csv.gz")
art["title"] = art.title.fillna(""); art["text"] = art.text.fillna("")

{'python': '3.12.3', 'seed': 42, 'split seed': 1, 'numpy': '2.5.4', 'pandas': '3.0.6', 'sklearn': '1.9.1', 'scipy': '1.18.1', 'torch': '2.5.1+cu121', 'transformers': '4.51.3', 'bitsandbytes': '0.50.2', 'trafilatura': '2.3.1'}


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


## Passes 1-2: links, e-mails, the outlet's own name

In [3]:
ruled, counts = AX.rules(art)
print("replacements:", counts)

replacements: {'email': 398, 'url': 8559, 'name': 17780}


## Pass 3: GLiNER (news outlets and journalists)

In [4]:
spans = AX.run(ruled, LOG)
print(f"spans: {len(spans):,} in {spans.article_id.nunique():,} articles ({spans.article_id.nunique() / len(art):.1%})")
text = {(a, f): t for f in ["title", "text"] for a, t in zip(ruled.article_id, ruled[f])}
spans["span"] = [text[(a, f)][s:e] for a, f, s, e in zip(spans.article_id, spans.field, spans.start, spans.end)]
for label, g in spans.groupby("label"):
    print(f"{label}: {len(g):,}; most frequent", g.span.value_counts().head(15).index.tolist())

spans: 93,099 in 20,415 articles (80.0%)
journalist: 36,796; most frequent ['I', 'reporter', 'Megyn', 'Jesse Valentine', 'Wyatt', 'reporters', 'Charlie Kirk', 'journalists', 'Tucker Carlson', 'Vance', 'Anthony Brian Logan', 'Bonnie Fuller', 'Carlson', 'JVL', 'she']
news outlet: 56,303; most frequent ['[OUTLET]', 'OUTLET', 'CNN', 'Fox News', 'Politico', 'Reuters', 'New York Times', 'BBC', 'The New York Times', 'MS NOW', 'Associated Press', 'Axios', 'CBS News', 'AP', 'The Associated Press']


In [5]:
anon = AX.apply(ruled, spans)
print("spans kept (no placeholder, a capitalized name):"); print(spans.groupby("label").kept.agg(["sum", "size"]).to_string())
for label, g in spans[spans.kept].groupby("label"):
    print(f"{label}: most frequent kept", g.span.value_counts().head(15).index.tolist())
print(f"articles touched by GLiNER: {spans[spans.kept].article_id.nunique() / len(art):.1%}")
before, after = AX.own_name_left(art), AX.own_name_left(anon)
print(f"articles with the name of their own outlet: before {before['own name left'].mean():.1%}, after {after['own name left'].mean():.1%} "
      f"({after['own name left'].sum():,} articles)")
print("outlets with the most leftovers:", after.groupby("domain")["own name left"].sum().sort_values(ascending=False).head(10).to_dict())
anon[["article_id", "domain", "title", "date", "text", "score", "label", "level"]].to_csv(C.OUT / "articles_anonymous.csv.gz", index=False)
after.to_csv(C.OUT / "anonymization_check.csv", index=False)
print("saved", C.OUT / "articles_anonymous.csv.gz")

spans kept (no placeholder, a capitalized name):
               sum   size
label                    
journalist   33816  36796
news outlet  41278  56303
journalist: most frequent kept ['Megyn', 'Jesse Valentine', 'Wyatt', 'Charlie Kirk', 'Tucker Carlson', 'Vance', 'Anthony Brian Logan', 'Bonnie Fuller', 'Carlson', 'JVL', 'Tim Foley', 'JD Vance', 'Sarah Longwell', 'Mamdani', 'MATTHEW RICE']
news outlet: most frequent kept ['CNN', 'Fox News', 'Politico', 'Reuters', 'New York Times', 'BBC', 'The New York Times', 'MS NOW', 'Associated Press', 'Axios', 'CBS News', 'AP', 'The Associated Press', 'TWN', 'New York Post']
articles touched by GLiNER: 71.8%


articles with the name of their own outlet: before 37.8%, after 1.6% (403 articles)
outlets with the most leftovers: {'thebulwark.com': 139, 'time.com': 87, 'politics.co.uk': 56, 'theweek.com': 35, 'thenational.scot': 15, 'popular.info': 14, 'thenation.com': 13, 'thebureau.news': 12, 'governing.com': 10, 'reason.com': 6}


saved /home/fabio.pinelli/John/Dataset/tentativi/20_FromScratch/articles_anonymous.csv.gz
